# Buku Kerja Hari 4 — NumPy & Pandas

Pendamping **Modul 4**. Estimasi 6–8 jam. Prasyarat: Hari 1–3 (`pip install numpy pandas`).

**Tujuan:** memakai array NumPy (indexing, masking, broadcasting, agregasi), serta Pandas (Series/DataFrame, seleksi `loc/iloc`, filter, kolom baru, cleaning, `groupby`, `merge`, `pivot_table`, baca/tulis file) dan menyelesaikan lima mini project berjenjang.

| Bagian | Isi |
|---|---|
| A | NumPy (L1–L9) |
| B | Pandas dasar (L10–L14) |
| C | Cleaning & operasi lanjutan (L15–L19) |
| D | Lima mini project (P1–P5) |
| E | Rekap |

> **Notebook SOLUSI** — semua latihan sudah terjawab dan dieksekusi; gunakan untuk membandingkan dengan jawaban Anda.

In [1]:
# @title Jalankan sel ini lebih dulu (alat bantu pemeriksa jawaban)
from contextlib import contextmanager
from unittest.mock import patch

STRICT = True
_hasil = {}

@contextmanager
def periksa(kode):
    """Dipakai di sel 'Periksa jawaban': gagal -> pesan petunjuk; lolos -> tanda centang."""
    try:
        yield
        _hasil[kode] = True
        print(f"✅ {kode} benar!")
    except AssertionError as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum tepat: {e}")
        if STRICT: raise
    except Exception as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum bisa dijalankan: {type(e).__name__}: {e}")
        if STRICT: raise

@contextmanager
def masukan(*jawaban):
    """Simulasi input(): with masukan('Ani', '80'): ... -> input() mengembalikan 'Ani' lalu '80'."""
    with patch("builtins.input", side_effect=list(jawaban)):
        yield

def ringkasan(semua):
    """Cetak rekap latihan: ✅ benar, ❌ salah, ⏳ belum dicoba."""
    benar = 0
    for kode in semua:
        s = _hasil.get(kode)
        tanda = "⏳ belum dicoba" if s is None else ("✅ benar" if s else "❌ perlu diperbaiki")
        benar += bool(s)
        print(f"{kode:<6} {tanda}")
    print(f"\nSkor: {benar} dari {len(semua)} latihan benar")

import numpy as np
import pandas as pd
from pathlib import Path
WORK = Path("latihan_hari4")
WORK.mkdir(exist_ok=True)


## A. NumPy

**Ringkas:** `np.array` menyimpan data numerik seragam sehingga operasi vektor jauh lebih cepat dan ringkas daripada loop. `arr[baris, kolom]` untuk indexing, `arr[a:b, c:d]` untuk slicing, `arr[arr > 60]` untuk *boolean masking*. Operasi antar array berbeda bentuk memakai *broadcasting*. `axis=0` = turun (per kolom), `axis=1` = ke samping (per baris).

In [2]:
arr = np.array([[10, 20, 30, 40], [50, 60, 70, 80], [90, 100, 110, 120]])
print(arr.shape, arr.dtype)
print(arr[0, 2], arr[:, 1], arr[1:, :2])
print(arr[arr > 60])
print(arr.sum(axis=0), arr.sum(axis=1))
print(np.where(arr > 70, arr, -1))

(3, 4) int64
30 [ 20  60 100] [[ 50  60]
 [ 90 100]]
[ 70  80  90 100 110 120]
[150 180 210 240] [100 260 420]
[[ -1  -1  -1  -1]
 [ -1  -1  -1  80]
 [ 90 100 110 120]]


### Latihan L1 — Membuat dan membentuk ulang array

Buat `m` = array `0..11` yang dibentuk menjadi matriks **3 baris × 4 kolom** (`np.arange` + `reshape`). Lalu hitung `jumlah_kolom` (jumlah tiap kolom) dan `jumlah_baris` (jumlah tiap baris).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
m ← arange(12) lalu reshape(3, 4)
jumlah per kolom → sum(axis=0)   jumlah per baris → sum(axis=1)
```

</details>

In [3]:
m = np.arange(12).reshape(3, 4)
jumlah_kolom = m.sum(axis=0)
jumlah_baris = m.sum(axis=1)

In [4]:
# Periksa jawaban
with periksa('L1'):
    assert m.shape == (3, 4), f"shape: {m.shape}"
    assert jumlah_kolom.tolist() == [12, 15, 18, 21], f"jumlah_kolom: {jumlah_kolom}"
    assert jumlah_baris.tolist() == [6, 22, 38], f"jumlah_baris: {jumlah_baris}"

✅ L1 benar!


### Latihan L2 — Indexing dan slicing

Dari `arr`, isi: `elemen` (baris 0, kolom 2), `submat` (baris 0–1, kolom 1–2), dan `kolom_terakhir` (seluruh baris pada kolom terakhir).

In [5]:
arr = np.array([[10, 20, 30, 40], [50, 60, 70, 80], [90, 100, 110, 120]])
elemen = arr[0, 2]
submat = arr[0:2, 1:3]
kolom_terakhir = arr[:, -1]

In [6]:
# Periksa jawaban
with periksa('L2'):
    assert elemen == 30
    assert submat.tolist() == [[20, 30], [60, 70]], f"submat: {submat}"
    assert kolom_terakhir.tolist() == [40, 80, 120]

✅ L2 benar!


### Latihan L3 — Boolean masking

Dari `arr` (sama seperti L2), isi `besar` (semua elemen > 60) dan `kelipatan_20` (elemen yang habis dibagi 20).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
besar ← arr[arr > 60]
kelipatan_20 ← arr[arr % 20 == 0]
```

</details>

In [7]:
arr = np.array([[10, 20, 30, 40], [50, 60, 70, 80], [90, 100, 110, 120]])
besar = arr[arr > 60]
kelipatan_20 = arr[arr % 20 == 0]

In [8]:
# Periksa jawaban
with periksa('L3'):
    assert besar.tolist() == [70, 80, 90, 100, 110, 120]
    assert kelipatan_20.tolist() == [20, 40, 60, 80, 100, 120]

✅ L3 benar!


### Latihan L4 — np.where

Buat `hasil`: salinan `arr` dengan elemen ≤ 70 diganti `-1` dan elemen lain tetap (`np.where`).

In [9]:
arr = np.array([[10, 20, 30, 40], [50, 60, 70, 80], [90, 100, 110, 120]])
hasil = np.where(arr > 70, arr, -1)

In [10]:
# Periksa jawaban
with periksa('L4'):
    assert hasil.tolist() == [[-1, -1, -1, -1], [-1, -1, -1, 80], [90, 100, 110, 120]], f"hasil: {hasil}"

✅ L4 benar!


### Latihan L5 — Broadcasting: nilai akhir berbobot

Setiap baris `nilai` = `[UTS, UAS, Tugas]` satu mahasiswa. Dengan bobot `[0.3, 0.4, 0.3]`, hitung `nilai_akhir` (array 3 elemen) **tanpa loop** (perkalian broadcasting lalu `sum(axis=1)`).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
nilai * bobot  → setiap baris dikali bobot (broadcasting)
.sum(axis=1)   → jumlahkan per baris
```

</details>

In [11]:
nilai = np.array([[80, 90, 70], [60, 75, 85], [90, 95, 88]])
bobot = np.array([0.3, 0.4, 0.3])
nilai_akhir = (nilai * bobot).sum(axis=1)

In [12]:
# Periksa jawaban
with periksa('L5'):
    assert np.allclose(nilai_akhir, [81.0, 73.5, 91.4]), f"nilai_akhir: {nilai_akhir}"

✅ L5 benar!


### Latihan L6 — Statistik bagian tengah matriks

`a` adalah matriks 6×6 berisi 1..36. Ambil bagian tengah 4×4 (baris & kolom indeks 1–4) ke `tengah`, lalu hitung `rata`, `simpangan` (`std`).

In [13]:
a = np.arange(1, 37).reshape(6, 6)
tengah = a[1:5, 1:5]
rata = tengah.mean()
simpangan = tengah.std()

In [14]:
# Periksa jawaban
with periksa('L6'):
    assert tengah.shape == (4, 4)
    assert rata == 18.5, f"rata: {rata}"
    assert abs(simpangan - 6.5) < 0.5 and abs(simpangan - a[1:5, 1:5].std()) < 1e-9, f"simpangan: {simpangan}"

✅ L6 benar!


### Latihan L7 — Normalisasi min-max

Ubah `x` menjadi skala 0–1: `(x - min) / (max - min)`. Simpan di `x_norm`.

In [15]:
x = np.array([2.0, 4.0, 6.0, 8.0])
x_norm = (x - x.min()) / (x.max() - x.min())

In [16]:
# Periksa jawaban
with periksa('L7'):
    assert np.allclose(x_norm, [0, 1/3, 2/3, 1]), f"x_norm: {x_norm}"
    assert x_norm.min() == 0 and x_norm.max() == 1

✅ L7 benar!


### Latihan L8 — Korelasi dua variabel

Dengan `a` dan `b` (acak ber-seed), hitung koefisien korelasi Pearson ke `korelasi` memakai `np.corrcoef`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
np.corrcoef(a, b) menghasilkan matriks 2×2; ambil elemen [0, 1]
```

</details>

In [17]:
np.random.seed(42)
a = np.random.randn(100)
b = np.random.randn(100)
korelasi = np.corrcoef(a, b)[0, 1]

In [18]:
# Periksa jawaban
with periksa('L8'):
    assert isinstance(float(korelasi), float)
    assert abs(korelasi - np.corrcoef(a, b)[0, 1]) < 1e-12, f"korelasi: {korelasi}"
    assert -1 <= korelasi <= 1

✅ L8 benar!


### Latihan L9 — Isi diagonal

Mulai dari matriks identitas 5×5 `m = np.eye(5)`, isi diagonalnya dengan `[10, 20, 30, 40, 50]` (`np.fill_diagonal`).

In [19]:
m = np.eye(5)
np.fill_diagonal(m, [10, 20, 30, 40, 50])

In [20]:
# Periksa jawaban
with periksa('L9'):
    assert np.diag(m).tolist() == [10, 20, 30, 40, 50], f"diagonal: {np.diag(m)}"
    assert m.sum() == 150, "elemen di luar diagonal harus 0"

✅ L9 benar!


## B. Pandas — Series & DataFrame

**Ringkas:** `Series` = kolom berlabel; `DataFrame` = tabel. Seleksi: `df["kol"]`, `df.loc[baris_label, kolom]`, `df.iloc[posisi]`. Filter: `df[df["ipk"] >= 3.5]`, gabungkan kondisi dengan `&` / `|` (dan **kurung**). Kolom baru: `df["baru"] = ...`, `np.where`, `.apply`, `.map`.

In [21]:
df = pd.DataFrame({"nama": ["Ani", "Budi", "Citra"], "prodi": ["IF", "SI", "IF"], "ipk": [3.75, 3.2, 3.9]})
print(df.shape, df.columns.tolist())
print(df[df["ipk"] >= 3.5][["nama", "ipk"]])
print(df.loc[df["prodi"] == "IF", "nama"].tolist(), df.iloc[0, 0])

(3, 3) ['nama', 'prodi', 'ipk']
    nama   ipk
0    Ani  3.75
2  Citra  3.90
['Ani', 'Citra'] Ani


### Latihan L10 — Series

Buat `s` = Series `[85, 90, 78, 92]` berindeks `["Ani","Budi","Citra","Dedi"]`. Isi `tinggi` dengan elemen Series yang ≥ 90 dan `rata` dengan rata-rata Series.

In [22]:
s = pd.Series([85, 90, 78, 92], index=["Ani", "Budi", "Citra", "Dedi"])
tinggi = s[s >= 90]
rata = s.mean()

In [23]:
# Periksa jawaban
with periksa('L10'):
    assert s["Budi"] == 90 and len(s) == 4
    assert tinggi.index.tolist() == ["Budi", "Dedi"], f"tinggi: {tinggi}"
    assert rata == 86.25

✅ L10 benar!


### Latihan L11 — Membuat dan memeriksa DataFrame

Buat `df` dari kamus di bawah, lalu isi `n_baris`, `n_kolom`, dan `nama_kolom` (list), serta `ipk_rata` (rata-rata kolom `ipk`).

In [24]:
data = {
    "nama": ["Ani", "Budi", "Citra", "Dedi", "Eka", "Fajar"],
    "prodi": ["IF", "SI", "IF", "SI", "IF", "SI"],
    "ipk": [3.75, 3.20, 3.90, 2.85, 3.55, 3.10],
}
df = pd.DataFrame(data)
n_baris, n_kolom = df.shape
nama_kolom = df.columns.tolist()
ipk_rata = df["ipk"].mean()

In [25]:
# Periksa jawaban
with periksa('L11'):
    assert (n_baris, n_kolom) == (6, 3)
    assert nama_kolom == ["nama", "prodi", "ipk"]
    assert abs(ipk_rata - 20.35 / 6) < 1e-9, f"ipk_rata: {ipk_rata}"

✅ L11 benar!


### Latihan L12 — Seleksi dan filter

Dengan `df` dari L11: `cumlaude` = list **nama** mahasiswa ber-IPK ≥ 3.5; `if_tinggi` = DataFrame mahasiswa prodi IF **dan** IPK ≥ 3.5.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
gabungkan kondisi dengan & dan beri kurung pada tiap kondisi
```

</details>

In [26]:
df = pd.DataFrame({
    "nama": ["Ani", "Budi", "Citra", "Dedi", "Eka", "Fajar"],
    "prodi": ["IF", "SI", "IF", "SI", "IF", "SI"],
    "ipk": [3.75, 3.20, 3.90, 2.85, 3.55, 3.10],
})
cumlaude = df.loc[df["ipk"] >= 3.5, "nama"].tolist()
if_tinggi = df[(df["prodi"] == "IF") & (df["ipk"] >= 3.5)]

In [27]:
# Periksa jawaban
with periksa('L12'):
    assert cumlaude == ["Ani", "Citra", "Eka"], f"cumlaude: {cumlaude}"
    assert if_tinggi["nama"].tolist() == ["Ani", "Citra", "Eka"]
    assert isinstance(if_tinggi, pd.DataFrame)

✅ L12 benar!


### Latihan L13 — Kolom baru: status dan grade

Tambahkan kolom `status` (`"Cumlaude"` jika ipk ≥ 3.5, selain itu `"Reguler"`, pakai `np.where`) dan kolom `grade` memakai `.apply` dengan function `tentukan_grade(ipk)`: `A` (≥3.5), `B` (≥3.0), `C` (≥2.5), selain itu `D`.

In [28]:
df = pd.DataFrame({"nama": ["Ani", "Budi", "Dedi"], "ipk": [3.75, 3.20, 2.85]})

def tentukan_grade(ipk):
    if ipk >= 3.5:
        return "A"
    if ipk >= 3.0:
        return "B"
    if ipk >= 2.5:
        return "C"
    return "D"

df["status"] = np.where(df["ipk"] >= 3.5, "Cumlaude", "Reguler")
df["grade"] = df["ipk"].apply(tentukan_grade)

In [29]:
# Periksa jawaban
with periksa('L13'):
    assert df["status"].tolist() == ["Cumlaude", "Reguler", "Reguler"], f"status: {df['status'].tolist()}"
    assert df["grade"].tolist() == ["A", "B", "C"], f"grade: {df['grade'].tolist()}"

✅ L13 benar!


### Latihan L14 — Sorting

Urutkan `df` berdasarkan `ipk` menurun ke `urut`, dan ambil 2 baris dengan IPK tertinggi ke `top2` (`nlargest`).

In [30]:
df = pd.DataFrame({"nama": ["Ani", "Budi", "Citra", "Dedi"], "ipk": [3.5, 3.9, 3.2, 3.7]})
urut = df.sort_values("ipk", ascending=False)
top2 = df.nlargest(2, "ipk")

In [31]:
# Periksa jawaban
with periksa('L14'):
    assert urut["nama"].tolist() == ["Budi", "Dedi", "Ani", "Citra"], f"urut: {urut['nama'].tolist()}"
    assert top2["nama"].tolist() == ["Budi", "Dedi"]

✅ L14 benar!


## C. Cleaning dan Operasi Lanjutan

**Ringkas:** `df.isna().sum()` menghitung missing; `fillna`, `dropna`, `drop_duplicates` membersihkan; `pd.to_numeric(..., errors="coerce")` mengubah teks angka (yang gagal → NaN). `groupby(...).agg(...)` meringkas per kelompok; `merge(..., on=..., how=...)` menggabungkan tabel; `pivot_table` membuat tabel silang.

In [32]:
kotor = pd.DataFrame({"nama": ["Ani", None, "Budi"], "ipk": [3.7, np.nan, 3.2]})
print(kotor.isna().sum().to_dict())
print(kotor["ipk"].fillna(kotor["ipk"].median()).tolist())
print(kotor.groupby("nama")["ipk"].mean())

{'nama': 1, 'ipk': 1}
[3.7, 3.45, 3.2]
nama
Ani     3.7
Budi    3.2
Name: ipk, dtype: float64


### Latihan L15 — Missing values dan duplikat

Dari `kotor`: isi `ipk` yang kosong dengan **median**, isi `prodi` yang kosong dengan `"Unknown"`, lalu hapus baris duplikat penuh. Simpan hasil di `bersih`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
salin df → isi ipk dengan median → isi prodi dengan "Unknown" → drop_duplicates()
```

</details>

In [33]:
kotor = pd.DataFrame({
    "nama": ["Ani", "Budi", "Budi", "Dedi"],
    "ipk": [3.7, np.nan, np.nan, 2.8],
    "prodi": ["IF", "SI", "SI", None],
})
bersih = kotor.copy()
bersih["ipk"] = bersih["ipk"].fillna(bersih["ipk"].median())
bersih["prodi"] = bersih["prodi"].fillna("Unknown")
bersih = bersih.drop_duplicates()

In [34]:
# Periksa jawaban
with periksa('L15'):
    assert len(bersih) == 3, f"baris: {len(bersih)} (duplikat Budi harus hilang)"
    assert bersih.isna().sum().sum() == 0, "masih ada missing value"
    assert bersih["ipk"].tolist() == [3.7, 3.25, 2.8], f"ipk: {bersih['ipk'].tolist()}"
    assert bersih["prodi"].tolist() == ["IF", "SI", "Unknown"]

✅ L15 benar!


### Latihan L16 — Konversi tipe data

Kolom `umur` berisi teks, sebagian tidak valid. Ubah ke angka dengan `pd.to_numeric(..., errors="coerce")` dan simpan di kolom `umur_num`; lalu `n_tidak_valid` = jumlah NaN pada kolom itu.

In [35]:
df = pd.DataFrame({"umur": ["20", "21", "n/a", "19"]})
df["umur_num"] = pd.to_numeric(df["umur"], errors="coerce")
n_tidak_valid = int(df["umur_num"].isna().sum())

In [36]:
# Periksa jawaban
with periksa('L16'):
    assert df["umur_num"].iloc[[0, 1, 3]].tolist() == [20, 21, 19]
    assert np.isnan(df["umur_num"].iloc[2])
    assert n_tidak_valid == 1

✅ L16 benar!


### Latihan L17 — GroupBy

Dari `df`: `rata_prodi` = rata-rata `ipk` per `prodi` (Series), dan `ringkas` = DataFrame hasil `agg` per prodi dengan kolom `rata` (mean ipk), `n` (jumlah mahasiswa).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
df.groupby("prodi").agg(rata=("ipk", "mean"), n=("nama", "count"))
```

</details>

In [37]:
df = pd.DataFrame({
    "nama": ["Ani", "Budi", "Citra", "Dedi", "Eka"],
    "prodi": ["IF", "SI", "IF", "SI", "IF"],
    "ipk": [3.75, 3.20, 3.90, 2.85, 3.55],
})
rata_prodi = df.groupby("prodi")["ipk"].mean()
ringkas = df.groupby("prodi").agg(rata=("ipk", "mean"), n=("nama", "count"))

In [38]:
# Periksa jawaban
with periksa('L17'):
    assert abs(rata_prodi["IF"] - 3.7333333) < 1e-6 and abs(rata_prodi["SI"] - 3.025) < 1e-9
    assert ringkas.columns.tolist() == ["rata", "n"], f"kolom: {ringkas.columns.tolist()}"
    assert ringkas["n"].to_dict() == {"IF": 3, "SI": 2}

✅ L17 benar!


### Latihan L18 — Merge

Gabungkan `mahasiswa` dan `nilai` pada `nim`: `inner` (hanya yang ada di keduanya) dan `kiri` (semua mahasiswa; `skor` kosong bila tak ada nilai).

In [39]:
mahasiswa = pd.DataFrame({"nim": ["2201", "2202", "2203"], "nama": ["Ani", "Budi", "Citra"]})
nilai = pd.DataFrame({"nim": ["2201", "2202", "2204"], "skor": [85, 90, 78]})
inner = pd.merge(mahasiswa, nilai, on="nim", how="inner")
kiri = pd.merge(mahasiswa, nilai, on="nim", how="left")

In [40]:
# Periksa jawaban
with periksa('L18'):
    assert inner["nama"].tolist() == ["Ani", "Budi"], f"inner: {inner}"
    assert len(kiri) == 3 and kiri["skor"].isna().tolist() == [False, False, True], f"kiri: {kiri}"

✅ L18 benar!


### Latihan L19 — Pivot table dan CSV

Buat `pivot`: rata-rata `nilai` dengan `nama` sebagai baris dan `matkul` sebagai kolom. Lalu simpan `pivot` ke `latihan_hari4/pivot.csv` dan baca kembali ke `kembali` (`index_col=0`).

In [41]:
data = pd.DataFrame({
    "nama": ["Ani", "Ani", "Budi", "Budi"],
    "matkul": ["Python", "BD", "Python", "BD"],
    "nilai": [85, 90, 78, 88],
})
pivot = data.pivot_table(values="nilai", index="nama", columns="matkul", aggfunc="mean")
pivot.to_csv(WORK / "pivot.csv")
kembali = pd.read_csv(WORK / "pivot.csv", index_col=0)

In [42]:
# Periksa jawaban
with periksa('L19'):
    assert pivot.loc["Ani", "Python"] == 85 and pivot.loc["Budi", "BD"] == 88, f"pivot: {pivot}"
    assert (WORK / "pivot.csv").exists(), "file pivot.csv belum dibuat"
    assert kembali.loc["Budi", "Python"] == 78 and list(kembali.index) == ["Ani", "Budi"]

✅ L19 benar!


## D. Lima Mini Project (Berjenjang)

Setiap project dipecah menjadi function yang mudah diuji. Selesaikan P1–P2 (wajib), lalu P3–P5 (tantangan).

### Latihan P1 — Project 1 — Kalkulator Nilai Akhir (Dasar)

Buat `hitung_nilai(df)` yang menerima DataFrame berkolom `nilai_uts`, `nilai_uas`, `nilai_tugas` dan mengembalikan **salinan** dengan kolom baru `nilai_akhir` (30% UTS + 40% UAS + 30% Tugas) dan `grade` (`A` ≥85, `B` ≥70, `C` ≥55, `D` ≥40, selain itu `E`). Buat juga `top_n(df, n)` → n baris dengan `nilai_akhir` tertinggi.

In [43]:
def hitung_nilai(df):
    hasil = df.copy()
    hasil["nilai_akhir"] = hasil["nilai_uts"] * 0.3 + hasil["nilai_uas"] * 0.4 + hasil["nilai_tugas"] * 0.3

    def grade(n):
        if n >= 85: return "A"
        if n >= 70: return "B"
        if n >= 55: return "C"
        if n >= 40: return "D"
        return "E"

    hasil["grade"] = hasil["nilai_akhir"].apply(grade)
    return hasil

def top_n(df, n):
    return df.nlargest(n, "nilai_akhir")

In [44]:
# Periksa jawaban
with periksa('P1'):
    d = pd.DataFrame({
        "nama": ["Ani", "Budi", "Citra", "Dedi"],
        "nilai_uts": [80, 60, 90, 30],
        "nilai_uas": [90, 75, 95, 40],
        "nilai_tugas": [70, 85, 88, 50],
    })
    h = hitung_nilai(d)
    assert "nilai_akhir" not in d.columns, "df asli tidak boleh diubah (gunakan salinan)"
    assert np.allclose(h["nilai_akhir"], [81.0, 73.5, 91.4, 40.0]), f"nilai_akhir: {h['nilai_akhir'].tolist()}"
    assert h["grade"].tolist() == ["B", "B", "A", "D"], f"grade: {h['grade'].tolist()}"
    assert top_n(h, 2)["nama"].tolist() == ["Citra", "Ani"]

✅ P1 benar!


### Latihan P2 — Project 2 — Cleaning Dataset Kotor

Buat `bersihkan(df)` untuk DataFrame berkolom `nama`, `ipk`, `prodi`, `umur`: (1) `strip` spasi di `nama`, (2) `nama` kosong → `"Unknown"`, (3) `ipk` kosong → median, (4) `prodi` kosong → `"Unknown"`, (5) `umur` → angka (`to_numeric`, gagal → NaN), (6) buang duplikat penuh **setelah** langkah 1–5. Kembalikan salinan; reset index.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
urutan penting: strip dulu → baru drop_duplicates (agar "Ani" == " Ani ")
```

</details>

In [45]:
def bersihkan(df):
    d = df.copy()
    d["nama"] = d["nama"].fillna("Unknown").str.strip()
    d["ipk"] = d["ipk"].fillna(d["ipk"].median())
    d["prodi"] = d["prodi"].fillna("Unknown")
    d["umur"] = pd.to_numeric(d["umur"], errors="coerce")
    return d.drop_duplicates().reset_index(drop=True)

In [46]:
# Periksa jawaban
with periksa('P2'):
    kotor = pd.DataFrame({
        "nama": ["Ani", " Ani ", None, "Dedi"],
        "ipk": [3.7, 3.7, 3.9, np.nan],
        "prodi": ["IF", "IF", "IF", None],
        "umur": ["20", "20", "n/a", "22"],
    })
    b = bersihkan(kotor)
    assert len(b) == 3, f"baris: {len(b)} — 'Ani' dan ' Ani ' harus dianggap duplikat setelah strip"
    assert b["nama"].tolist() == ["Ani", "Unknown", "Dedi"], f"nama: {b['nama'].tolist()}"
    assert b["ipk"].tolist() == [3.7, 3.9, 3.7], f"ipk: {b['ipk'].tolist()}"
    assert b["prodi"].tolist() == ["IF", "IF", "Unknown"]
    assert b["umur"].iloc[0] == 20 and np.isnan(b["umur"].iloc[1])
    assert kotor["nama"].iloc[1] == " Ani ", "df asli tidak boleh diubah"

✅ P2 benar!


### Latihan P3 — Project 3 — Merge Multi-Tabel (Menengah)

Buat `gabung_dan_ranking(mahasiswa, nilai, kehadiran)` yang meng-merge ketiga tabel pada `nim`, menambah kolom `rank` (peringkat `nilai_akhir`, 1 = tertinggi, bertipe int), dan mengembalikan DataFrame **terurut menurut rank**. Buat juga `korelasi_hadir_nilai(df)` → korelasi `persen_hadir` dan `nilai_akhir`.

In [47]:
def gabung_dan_ranking(mahasiswa, nilai, kehadiran):
    df = mahasiswa.merge(nilai, on="nim").merge(kehadiran, on="nim")
    df["rank"] = df["nilai_akhir"].rank(ascending=False).astype(int)
    return df.sort_values("rank").reset_index(drop=True)

def korelasi_hadir_nilai(df):
    return df["persen_hadir"].corr(df["nilai_akhir"])

In [48]:
# Periksa jawaban
with periksa('P3'):
    mhs = pd.DataFrame({"nim": ["2201", "2202", "2203", "2204", "2205"], "nama": ["Ani", "Budi", "Citra", "Dedi", "Eka"], "prodi": ["IF", "SI", "IF", "SI", "IF"]})
    nil = pd.DataFrame({"nim": ["2201", "2202", "2203", "2204", "2205"], "nilai_akhir": [85, 78, 92, 70, 88]})
    hdr = pd.DataFrame({"nim": ["2201", "2202", "2203", "2204", "2205"], "persen_hadir": [95, 70, 90, 60, 85]})
    g = gabung_dan_ranking(mhs, nil, hdr)
    assert g["nama"].tolist() == ["Citra", "Eka", "Ani", "Budi", "Dedi"], f"urutan: {g['nama'].tolist()}"
    assert g["rank"].tolist() == [1, 2, 3, 4, 5]
    assert g.columns.tolist()[:3] == ["nim", "nama", "prodi"]
    r = korelasi_hadir_nilai(g)
    assert 0.8 < r <= 1.0, f"korelasi: {r}"

✅ P3 benar!


### Latihan P4 — Project 4 — Pivot dan Reporting (Menengah–Tinggi)

Buat `laporan(data)` untuk DataFrame `nama`, `matkul`, `nilai`: pivot rata-rata `nilai` (baris `nama`, kolom `matkul`), tambah kolom `rata_rata` (rata-rata semua matkul per baris) dan `status` (`"Lulus"` jika `rata_rata >= 75`, selain itu `"Tidak Lulus"`). Kembalikan DataFrame pivot tersebut.

In [49]:
def laporan(data):
    pivot = data.pivot_table(values="nilai", index="nama", columns="matkul", aggfunc="mean")
    pivot["rata_rata"] = pivot.mean(axis=1)
    pivot["status"] = np.where(pivot["rata_rata"] >= 75, "Lulus", "Tidak Lulus")
    return pivot

In [50]:
# Periksa jawaban
with periksa('P4'):
    data = pd.DataFrame({
        "nama": ["Ani", "Ani", "Budi", "Budi", "Citra", "Citra"],
        "matkul": ["Python", "BD", "Python", "BD", "Python", "BD"],
        "nilai": [85, 90, 70, 75, 92, 85],
    })
    p = laporan(data)
    assert set(["BD", "Python", "rata_rata", "status"]).issubset(p.columns), f"kolom: {p.columns.tolist()}"
    assert p.loc["Ani", "rata_rata"] == 87.5 and p.loc["Budi", "rata_rata"] == 72.5
    assert p["status"].to_dict() == {"Ani": "Lulus", "Budi": "Tidak Lulus", "Citra": "Lulus"}, p["status"].to_dict()

✅ P4 benar!


### Latihan P5 — Project 5 — Ringkasan EDA (Tinggi)

Buat `ringkasan_eda(df)` yang mengembalikan dict dengan kunci: `baris`, `kolom`, `missing` (dict kolom → jumlah NaN, **hanya** kolom yang punya missing), `duplikat` (jumlah baris duplikat), `numerik` (list nama kolom numerik), `kategorik` (list nama kolom non-numerik). Fungsi ini bisa dipakai pada dataset apa pun (mis. Titanic) — di sini diuji dengan data kecil.

In [51]:
def ringkasan_eda(df):
    miss = df.isna().sum()
    return {
        "baris": len(df),
        "kolom": df.shape[1],
        "missing": {k: int(v) for k, v in miss.items() if v > 0},
        "duplikat": int(df.duplicated().sum()),
        "numerik": df.select_dtypes(include="number").columns.tolist(),
        "kategorik": df.select_dtypes(exclude="number").columns.tolist(),
    }

In [52]:
# Periksa jawaban
with periksa('P5'):
    d = pd.DataFrame({"nama": ["Ani", "Budi", "Budi", None], "umur": [20, 21, 21, np.nan], "prodi": ["IF", "SI", "SI", "IF"]})
    r = ringkasan_eda(d)
    assert r["baris"] == 4 and r["kolom"] == 3
    assert r["missing"] == {"nama": 1, "umur": 1}, f"missing: {r['missing']}"
    assert r["duplikat"] == 1
    assert r["numerik"] == ["umur"], f"numerik: {r['numerik']}"
    assert r["kategorik"] == ["nama", "prodi"], f"kategorik: {r['kategorik']}"

✅ P5 benar!


### Coba pada dataset sungguhan (opsional, perlu internet)

Jalankan `ringkasan_eda` pada dataset Titanic dari seaborn setelah **P5** lulus.

In [53]:
try:
    import seaborn as sns
    titanic = sns.load_dataset("titanic")
    hasil = ringkasan_eda(titanic)
    print(hasil)
except Exception as e:
    print("Dataset tidak dapat diunduh (perlu internet):", type(e).__name__)

{'baris': 891, 'kolom': 15, 'missing': {'age': 177, 'embarked': 2, 'deck': 688, 'embark_town': 2}, 'duplikat': 107, 'numerik': ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare'], 'kategorik': ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']}


## E. Rekap

1. Mengapa operasi vektor NumPy lebih baik daripada loop untuk data besar? Beri contoh dari latihan.
2. Apa beda `loc` dan `iloc`? Kapan masing-masing dipakai?
3. Satu masalah data (missing, duplikat, tipe salah) yang paling sering Anda temui dan cara mengatasinya:

> *Jawaban Anda:* …

**Checklist sebelum Hari 5**

- [ ] Semua latihan L1–L19 berstatus ✅
- [ ] Project P1 dan P2 lulus (P3–P5 sebagai tantangan)
- [ ] Saya bisa membaca dan menjelaskan `groupby`, `merge`, `pivot_table`
- [ ] Saya paham `axis=0` vs `axis=1` dan broadcasting

## Rekap hasil latihan

Jalankan sel di bawah setelah mengerjakan latihan.

In [54]:
ringkasan(['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10', 'L11', 'L12', 'L13', 'L14', 'L15', 'L16', 'L17', 'L18', 'L19', 'P1', 'P2', 'P3', 'P4', 'P5'])

L1     ✅ benar
L2     ✅ benar
L3     ✅ benar
L4     ✅ benar
L5     ✅ benar
L6     ✅ benar
L7     ✅ benar
L8     ✅ benar
L9     ✅ benar
L10    ✅ benar
L11    ✅ benar
L12    ✅ benar
L13    ✅ benar
L14    ✅ benar
L15    ✅ benar
L16    ✅ benar
L17    ✅ benar
L18    ✅ benar
L19    ✅ benar
P1     ✅ benar
P2     ✅ benar
P3     ✅ benar
P4     ✅ benar
P5     ✅ benar

Skor: 24 dari 24 latihan benar
